# Cue-aware pedestrian crossing intent and trajectory prediction: JAAD pipeline

Built from repo commit `a7289db`. One run: Stage A cue extraction on both GPUs (YOLO11m-pose, SegFormer-B2 Cityscapes), Stage B training of baselines, main model and ablations (3 seeds), results tables, surprise mining, and the end-to-end demo (YOLO11m-pose + ByteTrack).

Benchmarks use JAAD ground-truth boxes and track IDs; the detector and tracker are used only in the demo. Every number below is produced by this run.

In [ ]:
import base64, io, zipfile, os, sys
BUNDLE = '...'
CODE = '/kaggle/tmp/code'
os.makedirs(CODE, exist_ok=True)
zipfile.ZipFile(io.BytesIO(base64.b64decode(BUNDLE))).extractall(CODE)
print(sorted(os.listdir(CODE)))

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics', 'tabulate'], check=True)
import torch; print(torch.__version__, torch.cuda.device_count(), 'GPUs')

In [ ]:
import os, subprocess, sys
env = dict(os.environ, PYTHONPATH='/kaggle/tmp/code', PIPE_WORK='/kaggle/working')
p = subprocess.Popen([sys.executable, '-u', '-m', 'src.pipeline', 'all'], cwd='/kaggle/tmp/code', env=env,
                     stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in p.stdout:
    if 'it/s]' not in line and 'deprecated' not in line:
        print(line, end='')
p.wait(); print('pipeline exit code', p.returncode)

## Results
Tables are read from `results/tables` (generated by `src/eval/analysis.py` and `src/mining/surprise.py`).

In [ ]:
import glob, json, os
from IPython.display import Markdown, Image, Video, display
R = '/kaggle/working/results'
print(open(f'{R}/STATUS.json').read() if os.path.exists(f'{R}/STATUS.json') else 'no status')
for name in ('intent', 'trajectory', 'trajectory_best_of_k'):
    f = f'{R}/tables/{name}.md'
    if os.path.exists(f):
        display(Markdown(f'### {name}\n' + open(f).read()))
for f in ('surprise_summary.csv', 'error_intent_by_condition_main.csv', 'error_traj_fde_by_height_main.csv'):
    p = f'{R}/tables/{f}'
    if os.path.exists(p):
        import pandas as pd; display(Markdown(f'### {f}')); display(pd.read_csv(p))
for f in sorted(glob.glob(f'{R}/figures/*.png')):
    display(Markdown(f'### {os.path.basename(f)}')); display(Image(f, width=900))

## Demo clips
Clips are selected by the fixed rule in `configs/jaad.yaml` (`surprise`), including a failure case. Cue bars show what the model relied on, not why the person behaved as they did.

In [ ]:
import glob, os
from IPython.display import Markdown, Image, Video, display
D = '/kaggle/working/results/demo'
for f in sorted(glob.glob(f'{D}/*.mp4')):
    display(Markdown(f'**{os.path.basename(f)}** ({os.path.getsize(f) / 1e6:.1f} MB)'))
    display(Video(f, embed=True, width=800, html_attributes='controls muted'))
for f in sorted(glob.glob(f'{D}/*_storyboard.png')):
    display(Markdown(f'**{os.path.basename(f)}**')); display(Image(f, width=900))

## Optional (live kernel only)
An interactive cue toggle would need a running kernel with the trained models loaded; it is not part of the saved run. The counterfactual video above (`*_counterfactual_no_pose.mp4`) shows the same idea for one cue.